In [ ]:
import pandas as pd

# Pego la ruta del CSV y elijo cuales voy a usar.

ruta = "../data/raw/" 

orders    = pd.read_csv(ruta + "olist_orders_dataset.csv")
items     = pd.read_csv(ruta + "olist_order_items_dataset.csv")
customers = pd.read_csv(ruta + "olist_customers_dataset.csv")
products  = pd.read_csv(ruta + "olist_products_dataset.csv")
sellers   = pd.read_csv(ruta + "olist_sellers_dataset.csv")

for nombre, df in [("orders", orders), ("items", items), ("customers", customers), ("products", products), ("sellers", sellers)]: # desempaqueto con el bucle hasta sellers.
    print(nombre, df.shape)


orders (99441, 8)
items (112650, 7)
customers (99441, 5)
products (32951, 9)
sellers (3095, 4)


In [ ]:
for nombre, df in [("orders", orders), ("items", items), ("customers", customers), ("products", products), ("sellers", sellers)]:
    print(f"--- {nombre} ---")
    print("nulos:\n", df.isna().sum()[df.isna().sum() > 0])   # solo columnas con nulos por eso el > 0.
    print("filas duplicadas:", df.duplicated().sum())

# Claves unicas (tienen que dar True para poder unir sin duplicar)
print(orders["order_id"].is_unique)
print(customers["customer_id"].is_unique)
print(products["product_id"].is_unique)
print(sellers["seller_id"].is_unique)

# Resultado: claves unicas en True asi que puedo hacer el merge.
# Fe de errata: product_name_lenght y product_name_lenght, el LENGHT ya viene por defecto escrito asi en el dataset.

--- orders ---
nulos:
 order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64
filas duplicadas: 0
--- items ---
nulos:
 Series([], dtype: int64)
filas duplicadas: 0
--- customers ---
nulos:
 Series([], dtype: int64)
filas duplicadas: 0
--- products ---
nulos:
 product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64
filas duplicadas: 0
--- sellers ---
nulos:
 Series([], dtype: int64)
filas duplicadas: 0
True
True
True
True


In [ ]:
# Union: Juntamos los items con sus productos y vendedores y resumimos en una fila POR PEDIDO.
items_full = (items
    .merge(products, on="product_id", how="left", validate="m:1") #how=left MANTIENE los pedidos aunque no tengan items.
    .merge(sellers, on="seller_id", how="left", validate="m:1")) # validate = m:1(muchos a 1) hace que pandas tire error si un merge duplica filas.

items_por_pedido = items_full.groupby("order_id").agg(
    n_items=("order_item_id", "count"),
    precio_total=("price", "sum"),
    flete_total=("freight_value", "sum"),
    n_vendedores=("seller_id", "nunique"),
    peso_total_g=("product_weight_g", "sum"),   # ojo: da 0 si todos son nulos (afecta a muy pocos pedidos)
    categoria=("product_category_name", "first"), # first es que para cada PEDIDO toma el valor del PRIMER ITEM(tanto en categoria como estado del vendedor)
    estado_vendedor=("seller_state", "first"),
).reset_index()

In [ ]:
# Ahora unimos a los pedidos y a los clientes
df = (orders
      .merge(customers[["customer_id", "customer_state", "customer_city"]],
             on="customer_id", how="left", validate="m:1")
      .merge(items_por_pedido, on="order_id", how="left", validate="1:1")) # validate=1:1 en este caso. 

print(df.shape)

(99441, 17)


In [7]:
# Guardo el resultado de la union de mas arriba.

df.to_csv("../data/processed/pedidos_unidos.csv", index=False) # uso index=False para que pandas NO GUARDE una columna extra con el numero de fila.